# Non-GP probabilistic surrogates

Random Forest / Extra Trees / bootstrap boosting を同じ BoTorch posterior contract で扱う最小例です。ensemble のばらつきはモデルごとに意味が異なるため、calibrated Gaussian posterior としては解釈しません。

## このNotebookで学ぶこと

1. GP以外の確率的Surrogateでも、robotorchanの `fit()` / `posterior()` を使用する
2. アンサンブルの予測平均・ばらつきと、GPの事後分布の違いを理解する
3. BoTorchのMC獲得関数で**予測分布を使った候補提案**を行う
4. 非微分可能な木モデルでは、候補プール上の獲得関数を比較する
5. Random Forest・Extra Trees・Boosting・NGBoostの違いを確認する

**注意：** Tree ensembleの予測分散は、そのまま校正済みのベイズ的信用区間を意味しません。学習データの外挿にも注意が必要です。

In [ ]:
import torch
from robotorchan.models import RandomForestSurrogate

train_X = torch.linspace(0, 1, 20, dtype=torch.double).unsqueeze(-1)
train_Y = torch.sin(2 * torch.pi * train_X)
model = RandomForestSurrogate(train_X, train_Y, n_estimators=32, random_state=0)
model.fit()

### データと学習

`x∈[0,1]` に対する `sin(2πx)` の20点を観測します。`RandomForestSurrogate` は複数の決定木を学習し、その予測のばらつきを利用します。

最新のrobotorchanでは `RandomForestSurrogate.fit()` が公開されています。GPの `fit_gpytorch_mll()` をこのモデルに使用する必要はありません。

In [ ]:
test_X = torch.linspace(0, 1, 8, dtype=torch.double).unsqueeze(-1)
posterior = model.posterior(test_X)
posterior.mean.shape, posterior.variance.shape

### 事後平均とモデル間のばらつきを可視化

次の図ではRandom Forestの予測平均と、Posteriorの分散から計算した `mean ± 1.96*std` を表示します。**この帯は95%の被覆率を保証する区間ではありません**。Tree ensembleのばらつきの目安です。

In [ ]:
import matplotlib.pyplot as plt

plot_X = torch.linspace(0, 1, 201, dtype=train_X.dtype).unsqueeze(-1)
with torch.no_grad():
    rf_posterior = model.posterior(plot_X)
    rf_mean = rf_posterior.mean.reshape(-1)
    rf_std = rf_posterior.variance.clamp_min(0).sqrt().reshape(-1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_X[:, 0], torch.sin(2 * torch.pi * plot_X[:, 0]), "--", label="true function")
ax.plot(plot_X[:, 0], rf_mean, label="RF mean")
ax.fill_between(plot_X[:, 0], rf_mean - 1.96 * rf_std,
                rf_mean + 1.96 * rf_std, alpha=0.2,
                label="ensemble spread (not calibrated 95% CI)")
ax.scatter(train_X[:, 0], train_Y[:, 0], s=18, label="observations")
ax.set(xlabel="x", ylabel="y", title="Random Forest surrogate")
ax.legend()
plt.show()

## MC acquisition

Non-GP ensemble posterior は sampleable なので MC acquisition と接続できます。sklearn tree の候補入力に対する勾配は利用できないため、最適化には gradient-free search を使います。

### MC獲得関数の考え方

`qExpectedImprovement` は、予測分布からサンプルを引き、現在の最良観測値 `best_f` を上回る改善量の期待値を近似します。

このNotebookでは **q=1** として、1回に1候補を提案します。木モデルの `posterior()` はBoTorchのMC獲得関数と接続できますが、木の分割により入力に対する勾配は一般に利用できません。

したがって、`optimize_acqf()` の勾配最適化ではなく、有限候補プール上で獲得関数値を比較します。

In [ ]:
from botorch.acquisition.monte_carlo import qExpectedImprovement

acqf = qExpectedImprovement(model=model, best_f=train_Y.max())
value = acqf(torch.tensor([[[0.35]]], dtype=torch.double))
value

## Boosting の注意

Gradient Boosting / HistGradientBoosting では boosting stage 自体を posterior sample にしません。robotorchan は bootstrap ごとに complete boosting model を学習し、その complete-model predictions を empirical posterior members とします。

### 事後平均最大化との違い

`posterior.mean` が最大の点を選ぶ方法は**活用**に偏ります。一方、MC EIは予測のばらつきも使うため、改善の可能性がある未知の領域も候補になり得ます。ただし、アンサンブル分散が校正されていない場合、獲得関数の判断にもその影響が及びます。

## 候補提案と評価

学習済みのRandom Forestに対して有限候補プール上のMC獲得関数を評価し、最良候補を選択します。Tree ensembleの非微分可能性を考慮し、勾配ベースの最適化は使用しません。


In [ ]:
torch.manual_seed(0)
candidate_pool = torch.linspace(0, 1, 101, dtype=torch.double).unsqueeze(-1)
with torch.no_grad():
    acquisition_values = acqf(candidate_pool.unsqueeze(-2)).reshape(-1)
best_index = acquisition_values.argmax()
candidate = candidate_pool[best_index].unsqueeze(0)
observed_value = torch.sin(2 * torch.pi * candidate)
assert candidate.shape == (1, 1)
print("Proposed candidate:", candidate)
print("Acquisition value:", acquisition_values[best_index])
print("Objective evaluation:", observed_value)


### 候補プールの探索と評価

`candidate_pool` は101点の等間隔グリッドです。各候補を `q=1` の形状 `(101,1,1)` に整形して獲得関数を評価し、最大の候補を選択します。最後に元の目的関数で観測値を取得します。

これは**有限候補に限定した最適化**です。連続区間全体での厳密な最適解を保証するものではありません。

## Extra Trees: 候補提案と評価

Extra Treesも非微分可能なensemble posteriorを返すため、有限候補プールでMC獲得関数を比較します。


In [ ]:
from robotorchan.models import ExtraTreesSurrogate

extra_trees = ExtraTreesSurrogate(
    train_X, train_Y, n_estimators=32, random_state=0
)
extra_trees.fit()
extra_trees_acqf = qExpectedImprovement(
    model=extra_trees, best_f=train_Y.max()
)
with torch.no_grad():
    extra_trees_values = extra_trees_acqf(candidate_pool.unsqueeze(-2)).reshape(-1)
extra_trees_candidate = candidate_pool[extra_trees_values.argmax()].unsqueeze(0)
extra_trees_observation = torch.sin(2 * torch.pi * extra_trees_candidate)
assert extra_trees_candidate.shape == (1, 1)
print("Extra Trees candidate:", extra_trees_candidate)
print("Extra Trees objective evaluation:", extra_trees_observation)


### Extra TreesとRandom Forestの違い

Random Forestは通常、分割候補の中から良い分割を選びます。Extra Treesは分割点にもより強いランダム性を導入します。その結果、平均予測とアンサンブルのばらつきが異なることがあります。

両者とも `fit()` → `posterior()` → MC獲得関数 → 候補評価という流れを使います。

## Boosting ensemble: 候補提案と評価

Gradient BoostingとHistogram Gradient Boostingを学習し、MC獲得関数を有限候補プール上で評価します。


In [ ]:
from robotorchan.models import GradientBoostingSurrogate, HistGradientBoostingSurrogate

boosting_models = {
    "Gradient Boosting": GradientBoostingSurrogate(
        train_X, train_Y, n_members=4, n_estimators=12, random_state=0
    ),
    "Histogram Gradient Boosting": HistGradientBoostingSurrogate(
        train_X, train_Y, n_members=4, max_iter=12, random_state=0
    ),
}
for name, boosting_model in boosting_models.items():
    boosting_model.fit()
    boosting_acqf = qExpectedImprovement(
        model=boosting_model, best_f=train_Y.max()
    )
    with torch.no_grad():
        boosting_values = boosting_acqf(candidate_pool.unsqueeze(-2)).reshape(-1)
    boosting_candidate = candidate_pool[boosting_values.argmax()].unsqueeze(0)
    boosting_observation = torch.sin(2 * torch.pi * boosting_candidate)
    assert boosting_candidate.shape == (1, 1)
    print(name, "candidate:", boosting_candidate)
    print(name, "objective evaluation:", boosting_observation)


### Boostingの不確実性の扱い

Boostingでは弱学習器を逐次追加して1つの予測器を構成します。**途中のboosting stageを独立したPosteriorサンプルとして扱うのは適切ではありません。**

robotorchanのBootstrap Boostingは、再標本化ごとに**完成したBoostingモデル**を学習し、それらの予測から経験的な分布を構成します。Random Forestの木単位のばらつきとは意味が異なります。

## NGBoost: 候補提案と評価（オプション）

NGBoostは追加依存関係です。`ngboost`がインストールされていない場合は、この例のみスキップします。


In [ ]:
from importlib.util import find_spec

if find_spec("ngboost") is None:
    print("NGBoost example skipped: install robotorchan[ngboost] to run it.")
else:
    from robotorchan.models import NGBoostSurrogate

    ngboost_model = NGBoostSurrogate(
        train_X, train_Y, n_estimators=20, random_state=0, verbose=False
    )
    ngboost_model.fit()
    from botorch.sampling.normal import IIDNormalSampler

    ngboost_acqf = qExpectedImprovement(
        model=ngboost_model,
        best_f=train_Y.max(),
        sampler=IIDNormalSampler(sample_shape=torch.Size([128])),
    )
    with torch.no_grad():
        ngboost_values = ngboost_acqf(candidate_pool.unsqueeze(-2)).reshape(-1)
    ngboost_candidate = candidate_pool[ngboost_values.argmax()].unsqueeze(0)
    ngboost_observation = torch.sin(2 * torch.pi * ngboost_candidate)
    assert ngboost_candidate.shape == (1, 1)
    print("NGBoost candidate:", ngboost_candidate)
    print("NGBoost objective evaluation:", ngboost_observation)


### NGBoostの位置づけ

NGBoostは予測分布のパラメータを直接学習する確率的Boostingです。Tree ensembleのメンバー間分散とは異なる方法で不確実性を表します。

NGBoostはオプション依存関係です。インストールされていない場合は既存セルの分岐でこの節だけスキップします。

### モデル間の予測比較

最後に、学習済みモデルのPosterior Meanを同じ入力断面に重ねます。予測曲線の違いを確認できますが、この図だけで予測区間の校正やBO性能を判断することはできません。

In [ ]:
comparison_models = {"Random Forest": model, "Extra Trees": extra_trees, **boosting_models}
if "ngboost_model" in globals():
    comparison_models["NGBoost"] = ngboost_model

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_X[:, 0], torch.sin(2 * torch.pi * plot_X[:, 0]), "k--", label="true")
for name, surrogate in comparison_models.items():
    with torch.no_grad():
        predicted = surrogate.posterior(plot_X).mean.reshape(-1)
    ax.plot(plot_X[:, 0], predicted, label=name)
ax.scatter(train_X[:, 0], train_Y[:, 0], s=14, color="gray", label="observations")
ax.set(xlabel="x", ylabel="posterior mean", title="Non-GP surrogate comparison")
ax.legend(fontsize=8)
plt.show()

## まとめ

| モデル | 学習 | 予測分布の考え方 | 候補提案 |
|---|---|---|---|
| Random Forest | `fit()` | 木の予測の経験的分布 | MC EI + 有限候補 |
| Extra Trees | `fit()` | ランダム化された木の予測の経験的分布 | MC EI + 有限候補 |
| Gradient Boosting | `fit()` | 完成したBoostingモデルのBootstrap ensemble | MC EI + 有限候補 |
| Histogram Gradient Boosting | `fit()` | 完成したBoostingモデルのBootstrap ensemble | MC EI + 有限候補 |
| NGBoost（任意） | `fit()` | 学習した予測分布 | MC EI + 有限候補 |

**学習→Posterior→獲得関数→候補提案→目的関数評価**までを扱います。MC獲得関数が利用できることと、Posteriorの不確実性が十分に校正されていることは別問題です。